In [30]:
import torch
import random
from typing import Tuple

In [31]:
def generate_pair_indices(
    Y: torch.Tensor,
    N: int,
    allow_repeat: bool = True,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    device = Y.device
    N = int(N)
    if N <= 0:
        return (torch.empty(0, dtype=torch.long, device=device),
                torch.empty(0, dtype=torch.long, device=device),
                torch.empty(0, dtype=torch.float32, device=device))

    uniques = torch.unique(Y)
    label_to_idx = {int(l.item()): (Y == l).nonzero(as_tuple=True)[0] for l in uniques}
    sizes = [idx.numel() for idx in label_to_idx.values()]
    max_similar = sum(max(0, n * (n - 1) // 2) for n in sizes)
    if len(sizes) >= 2:
        total = sum(sizes)
        sum_sq = sum(n * n for n in sizes)
        max_dissimilar = (total * total - sum_sq) // 2
    else:
        max_dissimilar = 0

    target_sim = N // 2
    target_dis = N - target_sim
    if len(sizes) < 2:
        target_dis = 0
        target_sim = min(N, max_similar) if not allow_repeat else N
    if not allow_repeat:
        target_sim = min(target_sim, max_similar)
        target_dis = min(target_dis, max_dissimilar)

    if target_sim <= 0 and target_dis <= 0:
        return (torch.empty(0, dtype=torch.long, device=device),
                torch.empty(0, dtype=torch.long, device=device),
                torch.empty(0, dtype=torch.float32, device=device))

    idx1_list, idx2_list, y_list = [], [], []

    # Similar pairs
    if target_sim > 0:
        for _ in range(target_sim):
            valid_labels = [k for k, v in label_to_idx.items() if v.numel() >= 2]
            if not valid_labels:
                break
            l = random.choice(valid_labels)
            v = label_to_idx[l]
            i1, i2 = random.sample(range(v.numel()), 2)
            idx1_list.append(v[i1].item())
            idx2_list.append(v[i2].item())
            y_list.append(1.0)

    # Dissimilar pairs
    if target_dis > 0 and len(label_to_idx) >= 2:
        labels_list = list(label_to_idx.keys())
        for _ in range(target_dis):
            if len(labels_list) < 2:
                break
            l1, l2 = random.sample(labels_list, 2)
            v1, v2 = label_to_idx[l1], label_to_idx[l2]
            if v1.numel() == 0 or v2.numel() == 0:
                continue
            i1 = v1[random.randrange(v1.numel())].item()
            i2 = v2[random.randrange(v2.numel())].item()
            idx1_list.append(i1)
            idx2_list.append(i2)
            y_list.append(0.0)

    if len(y_list) == 0:
        return (torch.empty(0, dtype=torch.long, device=device),
                torch.empty(0, dtype=torch.long, device=device),
                torch.empty(0, dtype=torch.float32, device=device))

    idx1 = torch.tensor(idx1_list, dtype=torch.long, device=device)
    idx2 = torch.tensor(idx2_list, dtype=torch.long, device=device)
    y = torch.tensor(y_list, dtype=torch.float32, device=device)
    return idx1, idx2, y


In [32]:
def generate_pairs(X, Y, N):
    if N % 2 != 0:
        raise ValueError("N must be an even number.")

    num_same_label_pairs = N // 2
    num_different_label_pairs = N // 2

    label_to_indices = {label.item(): (Y == label).nonzero(as_tuple=True)[0].tolist()
                        for label in torch.unique(Y)}

    X1, X2, labels = [], [], []

    # Similar pairs
    for _ in range(num_same_label_pairs):
        valid_labels = [l for l, idxs in label_to_indices.items() if len(idxs) > 1]
        if not valid_labels:
            break
        sampled_label = random.choice(valid_labels)
        indices = label_to_indices[sampled_label]
        idx1, idx2 = random.sample(indices, 2)
        X1.append(X[idx1])
        X2.append(X[idx2])
        labels.append(1)

    # Dissimilar pairs
    for _ in range(num_different_label_pairs):
        if len(label_to_indices) < 2:
            break
        label1, label2 = random.sample(list(label_to_indices.keys()), 2)
        idx1 = random.choice(label_to_indices[label1])
        idx2 = random.choice(label_to_indices[label2])
        X1.append(X[idx1])
        X2.append(X[idx2])
        labels.append(0)

    X1 = torch.stack(X1)
    X2 = torch.stack(X2)
    labels = torch.tensor(labels, dtype=torch.float32)
    return X1, X2, labels


In [33]:
# For reproducibility
random.seed(42)
torch.manual_seed(42)

# Toy data: 6 samples, 3 features, 3 classes
X = torch.arange(18, dtype=torch.float32).reshape(6, 3)
Y = torch.tensor([0, 0, 1, 1, 2, 2])
N = 6


In [34]:
print(X)
print()
print(Y)

tensor([[ 0.,  1.,  2.],
        [ 3.,  4.,  5.],
        [ 6.,  7.,  8.],
        [ 9., 10., 11.],
        [12., 13., 14.],
        [15., 16., 17.]])

tensor([0, 0, 1, 1, 2, 2])


In [35]:
# Run index-based version
random.seed(42)
idx1, idx2, y_idx = generate_pair_indices(Y, N, allow_repeat=True)

# Use those indices to form the same pair tensors
X1_ref, X2_ref = X[idx1], X[idx2]

# Run your version
random.seed(42)
X1_new, X2_new, y_new = generate_pairs(X, Y, N)


In [36]:
print(X1_ref)
print()
print(X1_new)

tensor([[12., 13., 14.],
        [15., 16., 17.],
        [ 0.,  1.,  2.],
        [15., 16., 17.],
        [ 0.,  1.,  2.],
        [12., 13., 14.]])

tensor([[12., 13., 14.],
        [15., 16., 17.],
        [ 0.,  1.,  2.],
        [15., 16., 17.],
        [ 0.,  1.,  2.],
        [12., 13., 14.]])


In [37]:
print(X2_ref)
print()
print(X2_new)

tensor([[15., 16., 17.],
        [12., 13., 14.],
        [ 3.,  4.,  5.],
        [ 0.,  1.,  2.],
        [12., 13., 14.],
        [ 3.,  4.,  5.]])

tensor([[15., 16., 17.],
        [12., 13., 14.],
        [ 3.,  4.,  5.],
        [ 0.,  1.,  2.],
        [12., 13., 14.],
        [ 3.,  4.,  5.]])


In [38]:
print(y_idx)
print()
print(y_new)

tensor([1., 1., 1., 0., 0., 0.])

tensor([1., 1., 1., 0., 0., 0.])


In [39]:
print("Index-based output:")
print("y_idx:", y_idx)
print("Pairs (indices):")
print(torch.stack([idx1, idx2], dim=1))

print("\nYour function output:")
print("y_new:", y_new)

# Check whether pairs correspond (ignoring order, since sampling is random)
def same_pair_set(A1, A2, B1, B2, labelsA, labelsB):
    setA = {(tuple(a1.tolist()), tuple(a2.tolist()), float(l)) for a1, a2, l in zip(A1, A2, labelsA)}
    setB = {(tuple(b1.tolist()), tuple(b2.tolist()), float(l)) for b1, b2, l in zip(B1, B2, labelsB)}
    return setA == setB

print("\nExact same pairs?", same_pair_set(X1_ref, X2_ref, X1_new, X2_new, y_idx, y_new))


Index-based output:
y_idx: tensor([1., 1., 1., 0., 0., 0.])
Pairs (indices):
tensor([[4, 5],
        [5, 4],
        [0, 1],
        [5, 0],
        [0, 4],
        [4, 1]])

Your function output:
y_new: tensor([1., 1., 1., 0., 0., 0.])

Exact same pairs? True
